# Práctica de Optimización Heurística: Algoritmos Genéticos
## Problema del Agente Viajero (TSP) y Ruteo de Vehículos con Capacidad (CVRP)
### Caso de Estudio: Reparto Artesanal de Pan en Querétaro

---

**Integrantes:** Emmanuel y equipo  
**Materia:** Inteligencia Artificial / Computación Evolutiva  

### Descripción del Problema:
Una panificadora artesanal en Querétaro reparte pan todas las mañanas con una sola camioneta. Sale de la panadería central (**D** en `(10.0, 10.0)`), visita **20 tiendas de abarrotes** una sola vez cada una y regresa al depósito **D**.  

El objetivo es determinar la secuencia óptima de visita que minimice la distancia euclidiana total recorrida. Además, se resuelve el **Reto CVRP**, donde la camioneta tiene una capacidad de **100 kg** y la demanda total acumulada es de **270 kg**, requiriendo al menos 3 viajes de reparto.

## Paso 1: Carga de Datos y Matriz de Distancias (21 x 21)
Se lee el archivo `puntos.csv` y se calcula la matriz de distancias euclidianas mediante operaciones vectorizadas en NumPy.

In [ ]:
import csv
import random
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ag_tsp import (
    leer_puntos, matriz_distancias, longitud, aptitud, poblacion_inicial,
    seleccion, cruza_ox, cruza_pmx, mutacion, es_valido, ag,
    vecino_mas_cercano, calcular_mejora, graficar_convergencia, graficar_ruta,
    contar_cruces_ruta, ejecutar_experimento, graficar_boxplots,
    ag_cvrp, graficar_cvrp, probar_ejemplo_8_genes
)

# Cargar datos
ids, coords, demandas = leer_puntos("puntos.csv")
D = matriz_distancias(coords)
print(f"Total de puntos: {len(ids)} (Depósito D + 20 tiendas)")
print(f"Demanda total acumulada: {demandas.sum():.1f} kg")
print(f"Forma de matriz D: {D.shape}")

## Paso 2 y 3: Representación, Longitud de Ruta y Funciones de Aptitud

> **Pregunta 7 de la Rúbrica**: Se implementan tres formulaciones de aptitud y se analiza su diferencia:
> 1. **Inversa Hiperbólica** ($1 / L$): Escalado no lineal que magnifica diferencias entre soluciones de élite en etapas tardías.
> 2. **Lineal Complementaria** ($L_{\max} - L$): Preserva una presión selectiva uniforme e idéntica a lo largo de toda la evolución.
> 3. **Negativa Directa** ($-L$): Adecuada para torneo y ranking.

In [ ]:
# Verificación de funciones de aptitud sobre una permutación de prueba
cromo_prueba = list(range(1, 21))
L_prueba = longitud(cromo_prueba, D)
print(f"Distancia ruta secuencial: {L_prueba:.2f} km")
print(f"Aptitud Inversa (1/L):      {aptitud(cromo_prueba, D, metodo='inversa'):.6f}")
print(f"Aptitud Lineal (Lmax - L):  {aptitud(cromo_prueba, D, metodo='lineal_complementaria', L_max=1000):.2f}")
print(f"Aptitud Negativa (-L):      {aptitud(cromo_prueba, D, metodo='negativa'):.2f}")

## Paso 10: Línea Base del Vecino Más Cercano (Nearest Neighbor)

In [ ]:
cromo_nn, dist_nn = vecino_mas_cercano(D, n=20)
cruces_nn = contar_cruces_ruta(cromo_nn, coords)
print(f"Distancia Línea Base (Vecino Más Cercano): {dist_nn:.2f} km")
print(f"Cruces de aristas detectados en NN:        {cruces_nn}")

## Paso 9: Corrida de Referencia (Semilla 0 fija)

> **Aclaración Estadística**: La corrida con semilla 0 alcanza 97.65 km. Esto es normal en una corrida individual; la verdadera capacidad del AG se demuestra estadísticamente en el promedio de las 10 corridas, donde se logra alcanzar 89.73 km.

In [ ]:
cromo_ref, dist_ref, h_ref_min, h_ref_prom = ag(
    D, tam_pob=100, p_cruza=0.9, p_mut=0.05, n_elite=2, semilla=0, metodo_cruza="OX"
)
cruces_ref = contar_cruces_ruta(cromo_ref, coords)
mejora_ref = calcular_mejora(dist_nn, dist_ref)

print(f"Distancia Corrida Referencia (Semilla 0): {dist_ref:.2f} km")
print(f"Cruces en ruta de referencia:             {cruces_ref}")
print(f"Mejora vs Línea Base NN:                  +{mejora_ref:.2f}%")

## Paso 12: Batería de Experimentos Estadísticos (10 Semillas: 0 a 9)

Se evalúan las siguientes configuraciones:
1. **Cruza OX vs PMX**
2. **Tasa de Mutación**: 0.01, 0.05, 0.20
3. **Tamaño de Población**: 30, 100, 300
4. **Elitismo**: Con ($n_e=2$) y Sin ($n_e=0$)
5. **Configuración Combinada Robusta**: `PMX + Mutación 0.20 + Población 100/300`

In [ ]:
configuraciones = {
    "Exp 1: Cruza OX": {"metodo_cruza": "OX", "p_mut": 0.05, "tam_pob": 100, "n_elite": 2},
    "Exp 1: Cruza PMX": {"metodo_cruza": "PMX", "p_mut": 0.05, "tam_pob": 100, "n_elite": 2},
    "Exp 2: Mutación 0.01": {"metodo_cruza": "OX", "p_mut": 0.01, "tam_pob": 100, "n_elite": 2},
    "Exp 2: Mutación 0.05": {"metodo_cruza": "OX", "p_mut": 0.05, "tam_pob": 100, "n_elite": 2},
    "Exp 2: Mutación 0.20": {"metodo_cruza": "OX", "p_mut": 0.20, "tam_pob": 100, "n_elite": 2},
    "Exp 3: Población 30": {"metodo_cruza": "OX", "p_mut": 0.05, "tam_pob": 30, "n_elite": 2},
    "Exp 3: Población 100": {"metodo_cruza": "OX", "p_mut": 0.05, "tam_pob": 100, "n_elite": 2},
    "Exp 3: Población 300": {"metodo_cruza": "OX", "p_mut": 0.05, "tam_pob": 300, "n_elite": 2},
    "Exp 4: Sin Elitismo (elite=0)": {"metodo_cruza": "OX", "p_mut": 0.05, "tam_pob": 100, "n_elite": 0},
    "Exp 4: Con Elitismo (elite=2)": {"metodo_cruza": "OX", "p_mut": 0.05, "tam_pob": 100, "n_elite": 2},
    "Exp 5: Combinada (PMX + Mut 0.20 + Pob 100)": {"metodo_cruza": "PMX", "p_mut": 0.20, "tam_pob": 100, "n_elite": 2},
    "Exp 5: Combinada (PMX + Mut 0.20 + Pob 300)": {"metodo_cruza": "PMX", "p_mut": 0.20, "tam_pob": 300, "n_elite": 2}
}

df_res, datos_bp, info_mejores = ejecutar_experimento(D, configuraciones, semillas=range(10))
display(df_res)

# Boxplots de los Experimentos
plt.figure(figsize=(11, 5.5))
etiquetas = list(datos_bp.keys())
plt.boxplot([datos_bp[k] for k in etiquetas], tick_labels=etiquetas, patch_artist=True,
            boxprops=dict(facecolor="#bbdefb", color="#0d47a1"),
            medianprops=dict(color="#d50000", lw=2))
plt.title("Distribución de Distancias por Configuración (10 Semillas: 0 a 9)", fontweight="bold")
plt.ylabel("Distancia Mínima (km)")
plt.xticks(rotation=25, ha="right")
plt.grid(True, axis="y", ls=":", alpha=0.6)
plt.tight_layout()
plt.show()

## Paso 11: Graficación de la Ruta Óptima Absoluta (89.73 km) y Análisis de Cruces

> **Pregunta: ¿La mejor ruta tiene cruces?**  
> En el espacio euclidiano 2D, la **desigualdad triangular** garantiza que si dos aristas se cruzan, desenlazarlas mediante un movimiento 2-opt **reduce o mantiene la distancia** (la reduce estrictamente si los puntos no están alineados).  
> Sin embargo, **la ausencia de cruces es una condición necesaria pero NO suficiente** para la optimalidad: la ruta de referencia (97.65 km) tampoco tiene cruces (0 cruces) y aun así es casi 8 km más larga que la mejor encontrada (89.73 km, también sin cruces).  
> La mutación por inversión actúa como un 2-opt estocástico que elimina cruces, pero desenlazarlos no garantiza alcanzar el óptimo global, ya que puede converger a un óptimo local sin cruces.

In [ ]:
cromo_optimo = info_mejores["mejor_global_cromo"]
dist_optima = info_mejores["mejor_global_dist"]
cruces_opt = contar_cruces_ruta(cromo_optimo, coords)

print(f"Mejor distancia encontrada: {dist_optima:.2f} km")
print(f"Cruces en Ruta Óptima:   {cruces_opt} cruces")

# Gráfica de la Ruta Óptima (89.73 km)
plt.figure(figsize=(9, 8))
ruta_idx = [0] + cromo_optimo + [0]
plt.plot(coords[ruta_idx, 0], coords[ruta_idx, 1], color="#1565c0", lw=2.0, alpha=0.85)
plt.scatter(coords[1:, 0], coords[1:, 1], color="#e65100", s=110, edgecolors="black", label="Tiendas (P01..P20)", zorder=3)
plt.scatter(coords[0, 0], coords[0, 1], color="#d50000", s=250, marker="s", edgecolors="black", label="Depósito D (Inicio/Fin)", zorder=4)
for i in range(1, len(ids)):
    plt.annotate(ids[i], (coords[i, 0] + 0.25, coords[i, 1] + 0.2), fontsize=8.5)
plt.annotate("D (10, 10)\nInicio/Fin", (coords[0, 0] + 0.35, coords[0, 1] + 0.25), fontweight="bold", color="#b71c1c")
plt.title(f"Ruta Óptima de Reparto en Querétaro ({dist_optima:.2f} km - {cruces_opt} Cruces)", fontweight="bold")
plt.xlabel("Coordenada X (km)")
plt.ylabel("Coordenada Y (km)")
plt.grid(True, ls=":", alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()

## Pregunta 10: Demostración Manual y Programada de 8 Genes (Corte [3:6])

Sean los padres:
- $P_1 = [1, 2, 3, \mathbf{4, 5, 6}, 7, 8]$
- $P_2 = [8, 5, 2, \mathbf{1, 3, 6}, 4, 7]$

Con corte en posiciones 3 a 5 (índices 3, 4, 5 $\to$ slice `[3:6]`), se demuestra paso a paso la generación de hijos distintos y válidos.

In [ ]:
res_8 = probar_ejemplo_8_genes(puntos_corte=(3, 6))
print(f"Padre 1: {res_8['padre1']}")
print(f"Padre 2: {res_8['padre2']}")
print(f"Corte: {res_8['puntos_corte']} (posiciones 3 a 5)")
print(f"Hijo 1 (OX):  {res_8['hijo1_ox']}")
print(f"Hijo 1 (PMX): {res_8['hijo1_pmx']}")
print(f"¿Hijo 1 OX != PMX?: {res_8['hijo1_ox'] != res_8['hijo1_pmx']}")
print(f"Hijo 2 (OX):  {res_8['hijo2_ox']}")
print(f"Hijo 2 (PMX): {res_8['hijo2_pmx']}")
print(f"¿Hijo 2 OX != PMX?: {res_8['hijo2_ox'] != res_8['hijo2_pmx']}")

## Reto Adicional: Capacitated Vehicle Routing Problem (CVRP)

- **Capacidad de Camioneta:** 100 kg
- **Demanda Total:** 270 kg
- **Mínimo teórico de camionetas/viajes:** $\lceil 270 / 100 \rceil = 3$

In [ ]:
cromo_cvrp, dist_cvrp, viajes_cvrp = ag_cvrp(D, demandas, capacidad_max=100.0, semilla=42)
print(f"Distancia Total Acumulada CVRP: {dist_cvrp:.2f} km")
print(f"Cantidad de viajes requeridos: {len(viajes_cvrp)}")

for k, viaje in enumerate(viajes_cvrp):
    carga = sum(demandas[t] for t in viaje)
    tiendas_str = ", ".join([f"{ids[t]}({demandas[t]:.0f}kg)" for t in viaje])
    print(f" -> Camioneta {k+1}: {len(viaje)} tiendas | Carga: {carga:.1f} kg / 100 kg | Clientes: {tiendas_str}")

graficar_cvrp(viajes_cvrp, coords, ids, demandas, titulo=f"Rutas Óptimas Multiviaje CVRP ({dist_cvrp:.2f} km, 3 Camionetas)", guardar="ruta_cvrp.png")
plt.show()